# Kev: etiquetado de reseñas

Ejemplo mínimo de cómo llamar a Kev, que corre en un pod de RunPod, usando `typesafe_sdk`.

**Antes de correr:**
1. `MODELS_API_KEY` en el `.env` de la raíz del repo.
2. Pod levantado con `python deploy_models.py up` (guarda la URL en `models/config.json`). Bajarlo con `python deploy_models.py down`.

In [10]:
import os
import json
from pathlib import Path

import httpx
from dotenv import load_dotenv

ROOT = Path("../..")  # raíz del repo, relativa a este notebook
load_dotenv(ROOT / ".env")

config = json.loads((ROOT / "models" / "config.json").read_text())
BASE_URL = config["kev"]["urls"]["base"]
API_KEY = os.environ["MODELS_API_KEY"]
BASE_URL

'https://ovt7m1wzlv033u-8000.proxy.runpod.net'

In [14]:
httpx.get(f"{BASE_URL}/ping", timeout=30).status_code

502

In [15]:
from typesafe_sdk import Choice, TypeSafeClient

client = TypeSafeClient(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="kev-latest",
    # el SDK comprime con zstd los bodies de más de 1 KB y kev.serve no los sabe leer
    compression=False,
    timeout=120,
)

In [ ]:
SCALE = {
    "1": "Very negative",
    "2": "Negative",
    "3": "Mixed or acceptable",
    "4": "Positive",
    "5": "Very positive",
    "N/A": "The review does not give enough evidence to rate this aspect",
}

questions = {
    "food": Choice(instructions="How does the review rate the food and drinks?", criteria=SCALE),
    "wait_time": Choice(instructions="How does the review rate the speed of service?", criteria=SCALE),
}

review = "The food was delicious, but it took an hour to arrive."
response = client.system_one(state=review, questions=questions)
response.answers